# Определение смещённого кусочка

In [ ]:
import cv2
import numpy as np
def get_image_blocks(image, size):
    height, width = image.shape[:2]
    blocks = [[0 for _ in range(height // size)] for _ in range(width // size)]
    if height % size or width % size:
        raise Exception("Невозможно поделить на одинаковые блоки")
    for x in range(0, width, size):
        for y in range(0, height, size):
            block = image[y:y + size, x:x + size]
            blocks[x // size][y // size] = block
    return blocks

frame1 = cv2.imread(r"1.bmp")
frame1_blocks = get_image_blocks(frame1, 8)
cv2.imwrite(r"block1.png", frame1_blocks[90][50])
frame2 = cv2.imread(r"2.bmp")
frame2_blocks = get_image_blocks(frame2, 8)
cv2.imwrite(r"block2.png", frame2_blocks[90][50])

True

# Вектор смещения с момощью квадратичной меры

$$
\text{SSD}(dx, dy) = \sum_{i=0}^{7} \sum_{j=0}^{7} \left( B[i, j] - R[i + dy, j + dx] \right)^2
$$

$$
\text{MSE} = \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{n} 
$$

$y_i$ — истинное значение (исходный блок);

$\hat{y}_i$ — сравниваемое значение (тестируемый на сходство блок);

𝑛  — количество элементов.

In [ ]:
frame1 = cv2.imread(r"1.bmp")
frame2 = cv2.imread(r"2.bmp")

best_dx, best_dy = 0, 0
dx, dy = 0, 0
size = 8
x, y = 90 * size, 50 * size
cv2.imwrite(r"poisk.png", frame2[y-32:y+32, x-32:x+32])
target_block = frame1[y:y+size, x:x+size]
search_radius = 32
min_ssd = np.sum((target_block - frame2[y : y + size, x : x + size]) ** 2)

for dx in range(-search_radius, search_radius+1):
    for dy in range(-search_radius, search_radius+1):
        if (x + dx >= 0 and x + dx + size <= frame1.shape[1]) and \
                (y + dy >= 0 and y + dy + size <= frame1.shape[0]):
            block2 = frame2[y + dy:y + dy + size, x + dx:x + dx + size]
            ssd = np.sum((target_block - block2) ** 2)

            if ssd < min_ssd:
                min_ssd = ssd
                best_dx, best_dy = dx, dy

print(f"Смещение блока: dx = {best_dx}, dy = {best_dy}")
print(f"Квадратичная мера ошибки: ssd = {(min_ssd / 64):1.4f}")
cv2.imwrite(r"resultat.png", \
            frame2[y + best_dy : y + best_dy + size, x + best_dx : x + best_dx + size])

Смещение блока: dx = 11, dy = -3
Квадратичная мера ошибки: ssd = 152.3281


True

# Вектор смещения с помощью локализованного поиска

In [ ]:
frame1 = cv2.imread(r"1.bmp")
frame2 = cv2.imread(r"2.bmp")

size = 8
x, y = 90 * size, 50 * size
h = frame2.shape[0]
w = frame2.shape[1]
target_block = frame1[y : y + size, x : x + size]
best_y, best_x = x, y
min_ssd = np.sum((target_block - frame2[y : y + size, x : x + size]) ** 2)

while True:
    improved = False
    for dy, dx in [(-1,0), (1,0), (0,-1), (0,1)]:
        new_y = best_y + dy
        new_x = best_x + dx

        if (0 <= new_y <= h - 8) and (0 <= new_x <= w - 8):
            candidate = frame2[new_y : new_y + size, new_x : new_x + size]
            new_ssd = np.sum((target_block - candidate) ** 2)

            if new_ssd < min_ssd:
                best_y, best_x = new_y, new_x
                min_ssd = new_ssd
                improved = True
                break  # как только нашли лучше, двигаемся туда

    if not improved:
        break  # локальный минимум найден

print(f"Смещение блока: dx = {best_dx}, dy = {best_dy}")
print(f"Квадратичная мера ошибки: ssd = {(min_ssd / 64):1.4f}")
cv2.imwrite(r"resultat-1.png", \
            frame2[y + best_dy : y + best_dy + size, x + best_dx : x + best_dx + size])


Смещение блока: dx = 11, dy = -3
Квадратичная мера ошибки: ssd = 320.0938


True